# Яндекс Книги: проверка гипотез и анализ A/B-теста

Вы предобработали данные в SQL, и теперь они готовы для проверки гипотезы в Python. Загрузите данные пользователей из Москвы и Санкт-Петербурга c суммой часов их активности из файла yandex_knigi_data.csv. Если работаете локально, скачать файл можно по ссылке.

Проверьте наличие дубликатов в идентификаторах пользователей. Сравните размеры групп, их статистики и распределение.

Напомним, как выглядит гипотеза: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

По результатам анализа данных подготовьте аналитическую записку, в которой опишите:

Выбранный тип t-теста и уровень статистической значимости.

Результат теста, или p-value.

Вывод на основе полученного p-value, то есть интерпретацию результатов.

Одну или две возможные причины, объясняющие полученные результаты.

## Цели и задачи проекта
**Цель проекта** - определить различия в длительности чтения и прослушивания книг в приложении Я.Книги для пользователей из Москвы и Санкт-петербурга и проверить гипотезу:

Пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

Нулевая гипотеза $H_0: \mu_{\text{СПб}} \leq \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге не больше, чем в Москве.

Альтернативная гипотеза $H_1: \mu_{\text{СПб}} > \mu_{\text{Москва}}$ <br> Среднее время активности пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

**Задачи проекта**:
- Проверить и предобработать данные.
- Сравнить статистики, размеры и распределние групп пользователей.
- Провести статистический тест для проверки гипотезы.


## Описание данных

Таблица `yandex_knigi_data.csv`:
* `Unnamed: 0` - индекс строки;
* `city` - название города (Москва или Санкт-Петербург);
* `puid` - уникальный идентефикатор пользователя;
* `hours` - суммарное время потребления контента пользователем (в часах).

## Содержимое проекта
- Предобработка данных
- Проверка распределения пользователей
- Подсчёт основной метрики
- Аналитическая записка


## 1. Загрузка данных и знакомство с ними

Загрузите данные пользователей из Москвы и Санкт-Петербурга c их активностью (суммой часов чтения и прослушивания) из файла `/datasets/yandex_knigi_data.csv`.

In [3]:
import pandas as pd
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
from statsmodels.stats.proportion import proportions_ztest

In [4]:
yk_data = pd.read_csv('/datasets/yandex_knigi_data.csv')
yk_data.head(10)

,Unnamed: 0,city,puid,hours
0,0,Москва,9668,26.167776
1,1,Москва,16598,82.111217
2,2,Москва,80401,4.656906
3,3,Москва,140205,1.840556
4,4,Москва,248755,151.326434
5,5,Москва,352567,8.206369
6,6,Москва,439493,0.857758
7,7,Москва,494541,0.035072
8,8,Москва,647235,12.000076
9,9,Москва,656480,0.973032


In [5]:
yk_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8784 entries, 0 to 8783
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  8784 non-null   int64  
 1   city        8784 non-null   object 
 2   puid        8784 non-null   int64  
 3   hours       8784 non-null   float64
dtypes: float64(1), int64(2), object(1)
memory usage: 274.6+ KB


### 1.1 Предобработка данных

In [6]:
yk_data = yk_data.drop(columns=['Unnamed: 0'])

In [7]:
yk_data.head()

,city,puid,hours
0,Москва,9668,26.167776
1,Москва,16598,82.111217
2,Москва,80401,4.656906
3,Москва,140205,1.840556
4,Москва,248755,151.326434


In [8]:
yk_data['puid'].duplicated().sum()

244

In [9]:
yk_data = yk_data.drop_duplicates(subset=['puid'])

In [10]:
yk_data['puid'].duplicated().sum()

0

In [11]:
yk_data['city'].unique()

array(['Москва', 'Санкт-Петербург'], dtype=object)

In [12]:
yk_data.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 8540 entries, 0 to 8783
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   city    8540 non-null   object 
 1   puid    8540 non-null   int64  
 2   hours   8540 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 266.9+ KB


## 2. Проверка гипотезы в Python

Гипотеза звучит так: пользователи из Санкт-Петербурга проводят в среднем больше времени за чтением и прослушиванием книг в приложении, чем пользователи из Москвы. Попробуйте статистически это доказать, используя одностороннюю проверку гипотезы с двумя выборками:

- Нулевая гипотеза H₀: Средняя активность пользователей в часах в двух группах (Москва и Санкт-Петербург) не различается.

- Альтернативная гипотеза H₁: Средняя активность пользователей в Санкт-Петербурге больше, и это различие статистически значимо.

### 2.1 Проверка распределения пользователей

In [13]:
msk = yk_data[yk_data['city'] == 'Москва']
spb = yk_data[yk_data['city'] == 'Санкт-Петербург']

In [14]:
msk_c = msk.shape[0]
spb_c = spb.shape[0]
p = abs(msk_c - spb_c) / spb_c * 100
print(f"""Пользователей в Москве {msk_c} 
Пользователей в Санкт-Петербурге {spb_c} 
Процентная разница в количестве пользователей {p:.2f}%""")

Пользователей в Москве 6234 
Пользователей в Санкт-Петербурге 2306 
Процентная разница в количестве пользователей 170.34%


### 2.1 Проверка пересечения пользователей

In [15]:

users_msk = set(msk['puid'])
users_spb = set(spb['puid'])

common_users = users_msk.intersection(users_spb)
len(common_users)

0

### 2.2 Подсчёт основной метрики

In [16]:
msk_mh = round(msk['hours'].mean(),2)
spb_mh = round(spb['hours'].mean(),2)

diff = round((spb['hours'].mean() - msk['hours'].mean()),2)

print(f"Среднее время потребления контента в Санкт_Петербурге: {spb_mh} часа")
print(f"Среднее время потребления контента в Москве: {msk_mh} часа")
print(f"Разница в показателях: {diff:.2f} часа")

Среднее время потребления контента в Санкт_Петербурге: 11.26 часа
Среднее время потребления контента в Москве: 10.88 часа
Разница в показателях: 0.38 часа


In [17]:
from scipy import stats

spb_hours = spb['hours']
msk_hours = msk['hours']


t_stat, p_value = stats.ttest_ind(spb_hours, msk_hours, equal_var=False, alternative='greater')

print(f"t-статистика: {t_stat:.4f}")
print(f"p-value: {p_value:.4f}")

alpha = 0.05
if p_value < alpha:
    print("Отвергаем нулевую гипотезу: разница статистически значима, в СПб читают больше.")
else:
    print("Не получилось отвергнуть нулевую гипотезу: значимых различий не обнаружено.")

t-статистика: 0.4028
p-value: 0.3436
Не получилось отвергнуть нулевую гипотезу: значимых различий не обнаружено.


### 3. Аналитическая записка
Для проверки гипотезы мы выбрали t-тест Уэлча, так как у нас наблюдается большая разница в размерах  выборок. Мы выбрали стандартный уровень значимости 0.05.

P-value оказался 0.3436 и этот результат значительно выше нашего уровня значимости 0.05, следовательно мы не можем отвергнуть нулевую теорию и подтвердить, что жители Питера читают и слушают книги больше чем жители Москвы.

Разница в среднем времени потребления контента может быть обусловлена стандартным отклонением внутри групп и не подтверждается статистическим тестом.

----

# Часть 2. Анализ результатов A/B-тестирования

Теперь вам нужно проанализировать другие данные. Представьте, что к вам обратились представители интернет-магазина BitMotion Kit, в котором продаются геймифицированные товары для тех, кто ведёт здоровый образ жизни. У него есть своя целевая аудитория, даже появились хиты продаж: эспандер со счётчиком и напоминанием, так и подстольный велотренажёр с Bluetooth.

В будущем компания хочет расширить ассортимент товаров. Но перед этим нужно решить одну проблему. Интерфейс онлайн-магазина слишком сложен для пользователей — об этом говорят отзывы.

Чтобы привлечь новых клиентов и увеличить число продаж, владельцы магазина разработали новую версию сайта и протестировали его на части пользователей. По задумке, это решение доказуемо повысит количество пользователей, которые совершат покупку.

Ваша задача — провести оценку результатов A/B-теста. В вашем распоряжении:

* данные о действиях пользователей и распределении их на группы,

* техническое задание.

Оцените корректность проведения теста и проанализируйте его результаты.

## 1. Опишите цели исследования.



Цели иссследования:
- Провести оценку распределния групп тестирования.
- Определить корректость проведения тестирования.
- Оценить результаты тестирования.

## 2. Загрузите данные, оцените их целостность.


In [18]:
participants = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_participants.csv')
events = pd.read_csv('https://code.s3.yandex.net/datasets/ab_test_events.zip',
                     parse_dates=['event_dt'], low_memory=False)

In [19]:
participants

,user_id,group,ab_test,device
0,0002CE61FF2C4011,B,interface_eu_test,Mac
1,001064FEAAB631A1,B,recommender_system_test,Android
2,001064FEAAB631A1,A,interface_eu_test,Android
3,0010A1C096941592,A,recommender_system_test,Android
4,001E72F50D1C48FA,A,interface_eu_test,Mac
...,...,...,...,...
14520,FFE7FC140521F5F6,A,interface_eu_test,PC
14521,FFEFC0E55C1CCD4F,A,interface_eu_test,PC
14522,FFF28D02B1EACBE1,B,recommender_system_test,PC
14523,FFF28D02B1EACBE1,B,interface_eu_test,PC


## 3. По таблице `ab_test_participants` оцените корректность проведения теста:

   3\.1 Выделите пользователей, участвующих в тесте, и проверьте:

   - соответствие требованиям технического задания,

   - равномерность распределения пользователей по группам теста,

   - отсутствие пересечений с конкурирующим тестом (нет пользователей, участвующих одновременно в двух тестовых группах).

3.1.1 Поверим соотвествие ТЗ

In [20]:
participants.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14525 entries, 0 to 14524
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   user_id  14525 non-null  object
 1   group    14525 non-null  object
 2   ab_test  14525 non-null  object
 3   device   14525 non-null  object
dtypes: object(4)
memory usage: 454.0+ KB


In [21]:
participants['group'].unique()

array(['B', 'A'], dtype=object)

In [22]:
participants['ab_test'].unique()

array(['interface_eu_test', 'recommender_system_test'], dtype=object)

In [23]:
int_test = participants[participants['ab_test'] == 'interface_eu_test']
 

users_a = set(int_test[int_test['group'] == 'A']['user_id'])
users_b = set(int_test[int_test['group'] == 'B']['user_id'])

intersect_users_group = users_a.intersection(users_b)
len(intersect_users_group)

0

В датасете только 2 значения в столбце `ab_test` - "interface_eu_test" и "recommender_system_test", в столбце `group` так же 2 значения в соответсвии с ТЗ, пересеченний групп А и Б в нашем тесте нет.

3.1.1 Проверяем, есть ли пользователи, учавствующие одновременно в двух тестах

In [24]:
users_1 = set(participants[participants['ab_test'] == 'interface_eu_test']['user_id'])
users_2 = set(participants[participants['ab_test'] == 'recommender_system_test']['user_id'])


intersect_users = users_1.intersection(users_2)
len(intersect_users)

887

In [25]:

other_test = participants[participants['ab_test'] == 'interface_eu_test']


clash_users = other_test[other_test['user_id'].isin(intersect_users)]


print(clash_users['group'].value_counts())

B    456
A    431
Name: group, dtype: int64


Мы нашли 887 пользователей, которые учавствуют и в том, и в другом тесте, такие пользователи могут помешать корректности оценки результатов, т.к 456 из них состоят в контрольной группе первого теста. Для корректости оценки результатов нам необходимо отфильтровать этих пользователей.

In [26]:
filtred = participants[
    (participants['ab_test'] == 'interface_eu_test')&
    (~participants['user_id'].isin(intersect_users))
]

3.1.2 Проверяем равномерность распределния пользователей по группам

In [28]:

group_a = filtred[filtred['group'] == 'A']
group_b = filtred[filtred['group'] == 'B']

In [29]:
a_count = group_a.shape[0]
b_count = group_b.shape[0]
p = abs(a_count - b_count) / b_count * 100
print(f"""Пользователей в группе А {a_count} 
Пользователей в группе Б {b_count} 
Процентная разница в количестве пользователей {p:.2f}%""")

Пользователей в группе А 4952 
Пользователей в группе Б 5011 
Процентная разница в количестве пользователей 1.18%


Пользователи достаточно равномерно распределны по группам, разница в 1.18 процентных пункта незначительна.

3\.2 Проанализируйте данные о пользовательской активности по таблице `ab_test_events`:

- оставьте только события, связанные с участвующими в изучаемом тесте пользователями;

In [30]:
test_data = filtred.merge(events, on='user_id', how='inner')

In [31]:
test_data.head(5)

,user_id,group,ab_test,device,event_dt,event_name,details
0,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:31,registration,-2.38
1,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:49,login,NaN
2,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:37:57,login,NaN
3,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-07 04:38:54,login,NaN
4,0002CE61FF2C4011,B,interface_eu_test,Mac,2020-12-08 22:15:35,login,NaN


Мы объединили датасеты и оставили только события для пользователей, которые учавствуют только в нашем тестировании.


In [32]:
test_data.duplicated().sum()

5741

In [33]:
test_data = test_data.drop_duplicates(subset=['user_id', 'event_dt', 'event_name'])

In [34]:
test_data.duplicated().sum()

0

In [35]:
test_data.duplicated(subset=['user_id', 'event_dt', 'event_name']).sum()

0

- определите горизонт анализа: рассчитайте время (лайфтайм) совершения события пользователем после регистрации и оставьте только те события, которые были выполнены в течение первых семи дней с момента регистрации;

In [36]:



registrations = test_data[test_data['event_name'] == 'registration'][['user_id', 'event_dt']]
registrations = registrations.rename(columns={'event_dt': 'reg_dt'})



test_data = test_data.merge(registrations, on='user_id', how='left')


test_data['lifetime'] = (test_data['event_dt'] - test_data['reg_dt']).dt.total_seconds() / (3600 * 24)


test_data_filtered = test_data[test_data['lifetime'] <= 7]


print(f"Строк до фильтрации: {len(test_data)}")
print(f"Строк после фильтрации: {len(test_data_filtered)}")

Строк до фильтрации: 67582
Строк после фильтрации: 58411


Оцените достаточность выборки для получения статистически значимых результатов A/B-теста. Заданные параметры:

- базовый показатель конверсии — 30%,

- мощность теста — 80%,

- достоверность теста — 95%.

In [37]:


alpha = 0.05  
power = 0.8  
p1 = 0.30 
p2 = 0.33 


effect_size = proportion_effectsize(p1, p2)


power_analysis = NormalIndPower()
sample_size = power_analysis.solve_power(
    effect_size = effect_size,
    power = power,
    alpha = alpha,
    ratio = 1
)

print(f"Необходимый размер выборки для каждой группы: {int(sample_size)}")

Необходимый размер выборки для каждой группы: 3761


Наши выборки достаточны для получения статистически значимых результатов A/B-теста. Необходимый расчитанный размер - 3761 человек в одной группе, у нас около 5000 человек в каждой группе.


- рассчитайте для каждой группы количество посетителей, сделавших покупку, и общее количество посетителей.

In [38]:

total_users = test_data_filtered.groupby('group')['user_id'].nunique()


buyers = test_data_filtered[test_data_filtered['event_name'] == 'purchase'].groupby('group')['user_id'].nunique()


results = pd.DataFrame({
    'visitors': total_users,
    'purchasers': buyers
})


results['conversion_rate'] = (results['purchasers'] / results['visitors'] * 100).round(2)

results

,visitors,purchasers,conversion_rate
group,,,
A,4952,1377,27.81
B,5011,1480,29.54


- сделайте предварительный общий вывод об изменении пользовательской активности в тестовой группе по сравнению с контрольной.

In [39]:

conversion = results['purchasers'] / results['visitors']


cr_a = conversion['A']
cr_b = conversion['B']


abs_diff = cr_b - cr_a


rel_diff = (cr_b / cr_a) - 1

print(f"Конверсия группы A: {cr_a:.2%}")
print(f"Конверсия группы B: {cr_b:.2%}")
print("-" * 30)
print(f"Абсолютная разница: {abs_diff:.2%} (п.п.)")
print(f"Относительный прирост: {rel_diff:.2%}")

Конверсия группы A: 27.81%
Конверсия группы B: 29.54%
------------------------------
Абсолютная разница: 1.73% (п.п.)
Относительный прирост: 6.21%


Гипотеза уже не может быть подтверждена, т.к значение конверсии выросло лишь на 1.73%, тогда как предполагался рост на минимумм 3%.

## 4. Проведите оценку результатов A/B-тестирования:

- Проверьте изменение конверсии подходящим статистическим тестом, учитывая все этапы проверки гипотез.

In [40]:

n_a = test_data_filtered[ 
    (test_data_filtered.group== 'A')
]['user_id'].nunique()
n_b = test_data_filtered[ 
    (test_data_filtered.group== 'B')
]['user_id'].nunique() 

m_a = test_data_filtered[ 
    (test_data_filtered.group=='A')&
    (test_data_filtered.event_name == 'purchase')
]['user_id'].nunique() 

m_b = test_data_filtered[ # Считаем кол-во успехов в группе A
    (test_data_filtered.group=='B')&
    (test_data_filtered.event_name == 'purchase')
]['user_id'].nunique()

p_a, p_b = m_a/n_a, m_b/n_b # Считаем доли успехов для каждой группы: A и B

if (p_a*n_a > 10)and((1-p_a)*n_a > 10)and(p_b*n_b > 10)and((1-p_b)*n_b > 10): # Проверяем предпосылку о достаточном количестве данных
    print('Предпосылка о достаточном количестве данных выполняется!')
else:
    print('Предпосылка о достаточном количестве данных НЕ выполняется!')

alpha =  0.05 # Устанавливаем уровень значимости


stat_ztest, p_value_ztest = proportions_ztest( # проводим Z-тест пропорций
    [m_a, m_b],
    [n_a, n_b],
    alternative= 'smaller'
)

print(f'pvalue={p_value_ztest}') # Выводим полученное p-value 

if p_value_ztest > alpha:
    print('Нулевая гипотеза находит подтверждение!')
else:
    print('Нулевая гипотеза не находит подтверждения!')



Предпосылка о достаточном количестве данных выполняется!
pvalue=0.028262547212292124
Нулевая гипотеза не находит подтверждения!


- Опишите выводы по проведённой оценке результатов A/B-тестирования. Что можно сказать про результаты A/B-тестирования? Был ли достигнут ожидаемый эффект в изменении конверсии?

В нашем эксперименте мы задействовали 9963 пользователя, 4952 в группе А и 5011 в группе Б.
Конверсия в покупку составила 27,81% в контрольной группе A и 29,54% в тестовой группе B. Абсолютный прирост составил 1,73 процентного пункта.

Значение p-value — 0,0283, что значительно ниже уровня значимости 0.05, поэтому мы можем отвергнуть нулевую теорию и подтвердить, что новая версия сайта оказала положительное влияние на конверсию в покупку пользователей.

Статистический результат и бизнес-цель различаются: рост конверсии статистически значим, но наблюдаемый прирост 1,73 процентного пункта ниже целевого значения 3 процентных пункта. Достижение бизнес-цели не подтверждено.

Я бы порекомендовал расчитать конверсию разбив пользователей по используемым устройствам, например, рост могла показать дестктопая версия сайта, в то время как мобильная версия оказалась менее удобной, это поможет нам доработать сайт. В конце концов результат был, хоть и не тот, который мы ожидали, мы могли б собрать отзывы пользователей естовой групппы и доработать новую версию нашего сайта.

Так же нам мы можем проанализировать воронку и найти ее "узкое место", где отваливается большинство пользователей. Может оказаться дело не в самом сайте, а в одном кокнретном месте, которое отталкивает пользователей.

<a class="anchor" id="1-bullet"></a>
<div style="border:solid Chocolate 2px; padding: 40px">
Илья, спасибо за работу. Осталось исправить несколько комментариев и можно идти к заказчику. Молодец! 
   
   
    Что необходимо доработать: 
    
    1. Удалить дубликаты по событиям.
   
    2. Рекомендую:
    - оставить рекомендации.
 
    
    Жду новую версию проекта!)
    
[В начало](#3-bullet)